## Assignment: Image recognition
- Alumno 1: Aritz Bermejo Canal
- Alumno 2: Guillermo Alonso Rello
- Alumno 3: Juan García Santos

The goals of the assignment are:
* Develop proficiency in using Tensorflow/Keras for training Neural Nets (NNs).
* Put into practice the acquired knowledge to optimize the parameters and architecture of a feedforward Neural Net (ffNN), in the context of an image recognition problem.
* Put into practice NNs specially conceived for analysing images. Design and optimize the parameters of a Convolutional Neural Net (CNN) to deal with previous task.
* Train popular architectures from scratch (e.g., GoogLeNet, VGG, ResNet, ...), and compare the results with the ones provided by their pre-trained versions using transfer learning.

Información sobre el dataset:
El dataset original son 846 imagenes tomadas por un satélite de diferentes lugares del mundo. 761 imágenes son para train y 85 para test. En ellas aparecen objetos de 60 categorías. Se anotó con rectángulos la posición donde aparece un objeto en cada una de las imágenes mediante el punto (x,y) y la altura y la anchura el rectángulo.
Nosotros no vamos a trabajar con las imágenes originales, sino con los recortes de todos los objetos presentes en ellas. Por cada rectángulo posicionado encima de cada objeto se ha creado una imagen nueva de tamaño 224x224(x3 por el RGB). Esto significa que las imagenes no tienen por qué guardar las proporciones exactas del objeto original (se ha podido achatar o ampliar). Los recortes que pertenecen al conjunto de train son aquellos que vienen de las imágenes del conjunto de train y lo mismo para el test. Las 60 categorías están muy desbalancedas, por lo que solo vamos a utilizar los recortes de las 12 categorías más comunes.
Follow the link below to download the classification data set  “xview_recognition”: [https://drive.upm.es/s/4oNHlRFEd71HXp4](https://drive.upm.es/s/4oNHlRFEd71HXp4)
Los archivos del dataset son 2 JSON y 2 carpetas de imágenes, 1 de cada para train y test. El JSON se usa a modo de índice, esencialmente contiene pares (nombre_imagen, categoría). La carpeta de imágenes tiene 1 subcarpeta por categoría de objetos, donde se encuentran todas las imágenes de objetos de ese tipo, cada una con un nombre_imagen único referenciado desde el JSON.

First challenge: use of ffNN. We need to decide:
* Number of layers and number of units in each layer of your NN.
* Optimization algorithm and parameters to train the network.
* Check the evolution of these parameters during the optimization by using a validation subset and decide when to stop training (note that the test data set can only be used to evaluate the model).

Estrategia de ejecuciones:
1º Decidir el tamaño de la red: número de capas ocultas y neuronas por capa. Probar ascendentemente hasta llegar a un máximo de 5 capas. En principio el número de neuronas debe ir decreciendo capa a capa.
Para ello utilizar inicialmente Adam, RELU y descubrir un buen learning rate. Además, utilizar siempre regularización L2 (o dropout, solo 1 a la vez [aunque en principio no se puede dropout]) + batch normalization + early stopping con el conjunto de validación. También hay que decidir el tamaño de este conjunto, en principio conviene el mismo que el de test (los profes han puesto 10%). Prestar especial atención a la clase helicóptero, que solo tiene 70, podría pasar que la dividir en train y validación no se repartan bien.
2º Una vez estimado el tamaño, decidir mejor algoritmo (probar los otros que no son ADAM), con sus hiperparámetros, y función de activación (probar variantes de RELU). Finalmente probar a usar bagging.

Second challenge: Improve previous ffNN using convolutional layers to deal with the “xview_recognition” benchmark. To this end, you must perform some experiments using several CNNs. You should compare the performance obtained by your own CNN architectures against different popular CNNs (e.g., AlexNet, VGG, ResNet, etc) trained from scratch.

You must produce a few slides in PDF format describing the problem, all architectures, the performance obtained with your model on the train/valid/test data sets, plots of the evolution of costs and classification performance. Describe also the process that you have followed to reach your solution.




In [1]:
import uuid
import numpy as np

class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([]) # Realmente solo hay 1 objeto por cada imagen

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [2]:
categories = {0: 'Cargo plane', 1: 'Helicopter', 2: 'Small car', 3: 'Bus', 4: 'Truck', 5: 'Motorboat', 6: 'Fishing vessel', 7: 'Dump truck', 8: 'Excavator', 9: 'Building', 10: 'Storage tank', 11: 'Shipping container'}
all_labels = np.array(['Cargo plane','Helicopter','Bus','Cargo plane'])
categories_inv = {}
for i in categories.keys():
    categories_inv[categories[i]] = i

for i in range(len(all_labels)):
    all_labels[i] = categories_inv[all_labels[i]]
all_labels

array(['0', '1', '3', '0'], dtype='<U11')

In [3]:
import warnings
import rasterio
import numpy as np

# Devuelve la matriz 3D con los bits de una imagen dado el nombre del archivo
def load_geoimage(filename):
    filename = '../input/xviewrecognitionaug/' + filename
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
    src_raster = rasterio.open(filename, 'r')
    # RasterIO to OpenCV (see inconsistencies between libjpeg and libjpeg-turbo)
    input_type = src_raster.profile['dtype']
    input_channels = src_raster.count # 3 canales (RGB)
    img = np.zeros((src_raster.height, src_raster.width, src_raster.count), dtype=input_type)
    for band in range(input_channels): # Rellenar toda la matriz 2D del canal k de la matriz
        img[:, :, band] = src_raster.read(band+1)
    return img

# Genera grupos de imágenes del tamaño del batch durante todo el proceso de entrenamiento
def generator_images(objs, batch_size, batch_size_aug, train_dataset_size, do_shuffle=False):
    while True:
        if do_shuffle:
            np.random.shuffle(objs)
        groups = [objs[i:i+batch_size] for i in range(0, len(objs), batch_size)]
        for group in groups: # 1 epoch
            images, labels = [], [] # Matrices de imagen y one-hot encoding de cada una del grupo
            for (filename, obj) in group: # 1 mini-batch
                # Load image
                images.append(load_geoimage(filename)/255) # Bits de la imagen normalizados
                probabilities = np.zeros(len(categories)) # Vector para one-hot enconding
                probabilities[list(categories.values()).index(obj.category)] = 1
                labels.append(probabilities)
            images = np.array(images).astype(np.float32)
            labels = np.array(labels).astype(np.float32)
            yield images, labels # Devolver todo el grupo
        if(batch_size < train_dataset_size/10): # Al final de la epoch
            batch_size = batch_size * batch_size_aug
            print(batch_size)

In [4]:
#### Funciones de train

import math
import datetime
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, BatchNormalization, Conv2D, MaxPooling2D
from tensorflow.keras.regularizers import L2
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

def print_params(date_time, l_rate, b_size, bs_aug_factor, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, 
                 regularization, reg_param, batch_norm: bool, patience_lr, patience_es):
    print('Model parameters')
    print('Datetime', date_time)
    if isinstance(l_rate, float):
        print('Learning rate (fijo)', l_rate)
    else:
        print('Cosine decay')
        print('Learning rate antes del warmup', l_rate.initial_learning_rate)
        #print('Numero de steps en warnup', l_rate.warmup_steps)
        #print('Learning rate después del warmup', l_rate.warmup_target) 
        print('Numero de decay steps coseno', l_rate.decay_steps)
        print('Minimo learning rate con decay coseno', l_rate.alpha)
    print('Batch size', b_size)
    print('Batch size augmentation', bs_aug_factor)
    print('Epochs', epochs)
    print('Optimizer', optimizer)
    print('Optimizer hiperparams', opt_hparams)
    print('Capas ocultas', neurons_hidden_layers)
    print('Función de activación', activ_fun)
    print('Regularización', regularization)
    print('Reg param', reg_param)
    print('Normalización batch', batch_norm)
    print('Patience learning rate on plateau', patience_lr)
    print('Patience early stopping', patience_es)

def get_callbacks(patience_lr, patience_es):
    # Callbacks
    current_time = datetime.datetime.now()
    time_string = current_time.strftime("%Y-%m-%d_%H-%M-%S") # Replace colons with underscores
    
    model_checkpoint = ModelCheckpoint(f"{time_string}.keras", monitor='val_accuracy', verbose=1, save_best_only=True)
    
    early_stop = EarlyStopping('val_accuracy', patience=patience_es, verbose=1)
    terminate = TerminateOnNaN()    
    tensorboard = TensorBoard(log_dir=f"./logs/{time_string}")
    
    if patience_lr > 0:
        reduce_lr = ReduceLROnPlateau('val_accuracy', factor=0.1, patience=patience_lr, verbose=1) # Reducir el learning rate si se estanca
        callbacks = [model_checkpoint, reduce_lr, early_stop, terminate, tensorboard]
    else:
        callbacks = [model_checkpoint, early_stop, terminate, tensorboard]
    
    
    return callbacks, time_string

"""
Función para generar red neuronal. Se puede elegir:
- Learning rate o decreasing schedule
- Batch size
- Factor de incremento del batch size (cuanto cambia por epoch)
- Max Epochs
- Optimizer: Adam, SGD con momento o momento de Nesterov
- Hiperparámetros específicos del optimizador (en un array)
- Número de neuronas en cada capa oculta (un array de tamaño número de capas ocultas)
- Función de activación en cada capa oculta (la misma en todas)
- Uso de regularización (L2 o Dropout)
- Uso de batch norm (Sí o No)
"""
def train_ffnn(l_rate, b_size, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, 
               regularization, reg_param, batch_norm: bool, 
               patience_lr, patience_es, class_weight, metric ,model = None):
    
    if model == None:
        model = Sequential()
        # Capa de entrada, una neurona por píxel
        model.add(Flatten(input_shape=(224, 224, 3)))
        if(batch_norm): # No puede tener regularización, solo batch norm
            model.add(BatchNormalization())
        model.add(Activation(activ_fun))
    
    else: # Si estamos después de la convolución
        model.add(Flatten())

    # Añadir capas intermedias
    for n in neurons_hidden_layers:
        if(regularization == 'L2'):
            model.add(Dense(n, kernel_regularizer=L2(reg_param))) # Capa oculta
        
        elif(regularization == 'Dropout'):
            model.add(Dense(n))
            model.add(Dropout(reg_param))
        
        else:
            model.add(Dense(n)) # Capa oculta
        
        if(batch_norm):
            model.add(BatchNormalization())
        
        model.add(Activation(activ_fun))

    # Capa final, una salida por categoría (12)
    model.add(Dense(len(categories)))
    model.add(Activation('softmax'))
    
    model.summary()

    # Use optimizer
    if(optimizer == 'Adam'):
        opt = Adam(learning_rate=l_rate, **opt_hparams)
    elif(optimizer == 'SGD'):
        opt = SGD(learning_rate=l_rate, **opt_hparams)
    
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=metric)

    callbacks, time_string = get_callbacks(patience_lr, patience_es)

    # Preparar los datos y los generadores según el batch size
    objs_train = [(ann.filename, obj) for ann in anns_train for obj in ann.objects]
    objs_valid = [(ann.filename, obj) for ann in anns_valid for obj in ann.objects]
    train_steps = math.ceil(len(objs_train)/b_size)
    valid_steps = math.ceil(len(objs_valid)/b_size)
    train_generator = generator_images(objs_train, b_size, 1, len(objs_train), do_shuffle=True) # randomizar para no sesgar batches con orden
    valid_generator = generator_images(objs_valid, b_size, 1, len(objs_train), do_shuffle=False) # no es necesario randomizar, solo se testean

    print_params(time_string, l_rate, b_size, 1, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, 
                 regularization, reg_param, batch_norm, patience_lr, patience_es)
    # TODO ACTUALIZAR BATCH SIZE
    #for epoch in range(epochs): # 1 vez por época para poder actualizar el batch size
    h = model.fit(train_generator, validation_data=valid_generator, 
                  steps_per_epoch=train_steps, validation_steps=valid_steps, 
                  epochs=epochs, callbacks=callbacks, verbose=1, 
                  class_weight = class_weight)
    return model, h


def train_cnn(cnn_structure, l_rate, b_size, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, 
              regularization, reg_param, batch_norm: bool, patience_lr, 
              patience_es, class_weight, metric):
    model = Sequential()
    
    c_params_layers = cnn_structure['convolutions_params_layers'] 
    normalizations = cnn_structure['norm'] 
    pooling = cnn_structure['pooling'] 
    p_params = cnn_structure['pooling_params'] 
    
    for i, c_params in enumerate(c_params_layers):
        if i == 0:
            model.add(Conv2D(input_shape=(224, 224, 3), 
                             filters=c_params[0], 
                             kernel_size=c_params[1], 
                             strides=c_params[2], 
                             padding=c_params[3]))
        else:
            model.add(Conv2D(filters=c_params[0], 
                             kernel_size=c_params[1], 
                             strides=c_params[2], 
                             padding=c_params[3]))
        
        if(normalizations[i]):
            model.add(BatchNormalization())
        model.add(Activation(activ_fun))
        if(pooling[i]):
            model.add(MaxPooling2D(p_params[0], strides=p_params[1], padding=p_params[2]))
    
    # Completa con la parte ffnn densa y entrena
    return train_ffnn(l_rate, b_size, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, 
                      regularization, reg_param, batch_norm, patience_lr, 
                      patience_es, class_weight, metric,model = model)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

In [5]:
#### Funciones de test

import matplotlib.pyplot as plt
import numpy as np

def test_net(model, test_anns):
    y_true, y_pred = [], []
    for ann in test_anns:
        # Load image
        image = load_geoimage(ann.filename) / 255
        for obj_pred in ann.objects:
            # Generate prediction
            warped_image = np.expand_dims(image, 0)
            predictions = model.predict(warped_image, verbose = 0)
            # Save prediction
            pred_category = list(categories.values())[np.argmax(predictions)]
            pred_score = np.max(predictions)
            y_true.append(obj_pred.category)
            y_pred.append(pred_category)
    return y_true, y_pred

def draw_confusion_matrix(cm, categories):
    # Draw confusion matrix
    fig = plt.figure(figsize=[6.4*pow(len(categories), 0.5), 4.8*pow(len(categories), 0.5)])
    ax = fig.add_subplot(111)
    cm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], np.finfo(np.float64).eps)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.get_cmap('Blues'))
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), xticklabels=list(categories.values()), yticklabels=list(categories.values()), ylabel='Annotation', xlabel='Prediction')
    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    # Loop over data dimensions and create text annotations
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], '.2f'), ha="center", va="center", color="white" if cm[i, j] > thresh else "black", fontsize=int(20-pow(len(categories), 0.5)))
    fig.tight_layout()
    plt.show(fig)

# Imprimir resultados a partir de la matriz de confusión
def print_results(cm):
    # Compute the accuracy
    correct_samples_class = np.diag(cm).astype(float)
    total_samples_class = np.sum(cm, axis=1).astype(float)
    total_predicts_class = np.sum(cm, axis=0).astype(float)
    print('Mean Accuracy: %.3f%%' % (np.sum(correct_samples_class) / np.sum(total_samples_class) * 100))
    acc = correct_samples_class / np.maximum(total_samples_class, np.finfo(np.float64).eps)
    print('Mean Recall: %.3f%%' % (acc.mean() * 100))
    acc = correct_samples_class / np.maximum(total_predicts_class, np.finfo(np.float64).eps)
    print('Mean Precision: %.3f%%' % (acc.mean() * 100))
    for idx in range(len(categories)):
        # True/False Positives (TP/FP) refer to the number of predicted positives that were correct/incorrect.
        # True/False Negatives (TN/FN) refer to the number of predicted negatives that were correct/incorrect.
        tp = cm[idx, idx]
        fp = sum(cm[:, idx]) - tp
        fn = sum(cm[idx, :]) - tp
        tn = sum(np.delete(sum(cm) - cm[idx, :], idx))
        # True Positive Rate: proportion of real positive cases that were correctly predicted as positive.
        recall = tp / np.maximum(tp+fn, np.finfo(np.float64).eps)
        # Precision: proportion of predicted positive cases that were truly real positives.
        precision = tp / np.maximum(tp+fp, np.finfo(np.float64).eps)
        # True Negative Rate: proportion of real negative cases that were correctly predicted as negative.
        specificity = tn / np.maximum(tn+fp, np.finfo(np.float64).eps)
        # Dice coefficient refers to two times the intersection of two sets divided by the sum of their areas.
        # Dice = 2 |A∩B| / (|A|+|B|) = 2 TP / (2 TP + FP + FN)
        f1_score = 2 * ((precision * recall) / np.maximum(precision+recall, np.finfo(np.float64).eps))
        print('> %s: Recall: %.3f%% Precision: %.3f%% Specificity: %.3f%% Dice: %.3f%%' % (list(categories.values())[idx], recall*100, precision*100, specificity*100, f1_score*100))

In [6]:
#### Funciones para cargar BD
import json
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Load JSON and then use it to get objects
def import_database(json_file):
    # Load database JSON.
    with open(json_file) as ifs:
        json_data = json.load(ifs)
    ifs.close()

    counts = dict.fromkeys(categories.values(), 0)
    anns = []
    all_labels = []
    # Crear un GenericImage por cada archivo con su GenericObject correspondiente categorizado dentro
    for json_img, json_ann in zip(json_data['images'].values(), json_data['annotations'].values()):
        image = GenericImage(json_img['filename'])
        image.tile = np.array([0, 0, json_img['width'], json_img['height']])
        obj = GenericObject()
        obj.bb = (int(json_ann['bbox'][0]), int(json_ann['bbox'][1]), int(json_ann['bbox'][2]), int(json_ann['bbox'][3]))
        obj.category = json_ann['category_id']
        all_labels.append(obj.category)
        # Resampling strategy to reduce training time
        counts[obj.category] += 1
        image.add_object(obj)
        anns.append(image)
    print(counts)
    labels_unique = np.array(list(categories.keys()))
    
    for i in range(len(all_labels)):
        all_labels[i] = categories_inv[all_labels[i]]
    all_labels = np.array(all_labels,dtype=int)
    
    class_weights = compute_class_weight('balanced',
                                    classes = labels_unique,
                                     y = all_labels)

    class_weight_dict = {label: weight 
                     for label,weight in zip(labels_unique, class_weights)}

    return anns, class_weight_dict

In [7]:
# Cargamos las BD de train y test
import os
print(os.listdir('/kaggle/input/'))
json_train_file = '../input/xviewrecognitionaug/xview_ann_train.json'
json_test_file = '../input/xviewrecognitionaug/xview_ann_test.json'
print('Train')
anns_train, class_weight_dict = import_database(json_train_file)
print('Test')
anns_test, _ = import_database(json_test_file)
class_weight_dict

['xviewrecognition', 'xviewrecognitionaug']
Train
{'Cargo plane': 4689, 'Helicopter': 4689, 'Small car': 4689, 'Bus': 4689, 'Truck': 4689, 'Motorboat': 4689, 'Fishing vessel': 4689, 'Dump truck': 4689, 'Excavator': 4689, 'Building': 4689, 'Storage tank': 4689, 'Shipping container': 4689}
Test
{'Cargo plane': 83, 'Helicopter': 1, 'Small car': 487, 'Bus': 242, 'Truck': 305, 'Motorboat': 394, 'Fishing vessel': 93, 'Dump truck': 122, 'Excavator': 57, 'Building': 542, 'Storage tank': 243, 'Shipping container': 66}


{0: 1.0,
 1: 1.0,
 2: 1.0,
 3: 1.0,
 4: 1.0,
 5: 1.0,
 6: 1.0,
 7: 1.0,
 8: 1.0,
 9: 1.0,
 10: 1.0,
 11: 1.0}

In [8]:
# Separar conjunto de train
from sklearn.model_selection import train_test_split
anns_train, anns_valid = train_test_split(anns_train, test_size=0.1, random_state=1, shuffle=True)
N_train = len(anns_train) # Size of train dataset (without valid)

In [9]:
# Crea un schedule de cosine decay

def get_lr_schedule(N_steps, percentage_warmup, initial_learning_rate, target_warm_learning_rate, final_learning_rate):
    warmup_steps = math.floor(percentage_warmup * N_steps) # Step = minibatch
    decay_steps = math.floor((1 - percentage_warmup) * N_steps) # Pasos de decay coseno
    l_rate_decay = CosineDecay(
        initial_learning_rate, decay_steps, alpha=final_learning_rate, 
        #warmup_target=target_warm_learning_rate, warmup_steps=warmup_steps
    )
    print('Warmup', warmup_steps, 'Decay', decay_steps)
    return l_rate_decay


In [10]:
# Permite crear varias redes haciendo listas de parametros
from tensorflow.keras.optimizers.schedules import CosineDecay

from tensorflow_addons.metrics import FBetaScore
import math
import numpy as np
from tensorflow import random

random.set_seed(120)

percentage_warmup = 0
initial_learning_rate = 0.1 # Antes del warmup
target_warm_learning_rate = 0.1 # Después del warmup
final_learning_rate = 1e-6 # el learning rate mínimo

cosine_decay_params = [percentage_warmup, initial_learning_rate, target_warm_learning_rate, final_learning_rate]

num_models = 1
b_size = [128] * num_models
N_epochs = [40] * num_models
l_rate = [cosine_decay_params, 0.1]
#l_rate = [0.01]
optimizer = ['SGD'] * num_models
#opt_hparams = [{'momentum':0.0, 'nesterov': False, 'weight_decay':None}]
opt_hparams = [{'momentum':0.9, 'nesterov': False, 'weight_decay':0.0005}] * num_models
neurons_hidden_layers = [[4096, 4096]] * num_models
activ_fun = ['relu'] * num_models
regularization = ['Dropout'] * num_models
#regularization = ['None']
reg_param = [0.5] * num_models
batch_norm = [True] * num_models
patience_lr = [3] * num_models
patience_es = [5] * num_models
class_weight = [None] * num_models
#class_weight = [class_weight_dict] * num_models
f1 = FBetaScore(num_classes = 12,average = 'weighted')
#metric = [f1]
metric = ['accuracy'] * num_models

convolutions_params_layers = [[64, 3, 1,'same'], 
                              [64, 3, 1,'same'],
                              [128, 3, 1,'same'],
                              [128, 3, 1,'same'],
                              [256, 3, 1,'same'],
                              [256, 3, 1,'same'],
                              [256, 3, 1,'same'],
                              [512, 3, 1,'same'],
                              [512, 3, 1,'same'],
                              [512, 3, 1,'same'],
                              [512, 3, 1,'same'],
                              [512, 3, 1,'same'],
                              [512, 3, 1,'same']]

cnn_structure = [{'convolutions_params_layers' : convolutions_params_layers, 
                 'norm' : [False, False, False, False, False, False, False, False, False, False, False, False, False], 
                 'pooling' : [False, True, False, True, False, False, True, False, False, True, False, False, True], 
                 'pooling_params' : [2, 2, 'valid']}] * num_models

/opt/conda/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


#### Report

You must prepare a report (PDF) describing:
* The problems and data sets (briefly).
* The process that you have followed to reach your solution for the “xview_recognition” benchmark, including your intermediate results. You must discuss and compare these results properly.
* Final network architectures, including optimization algorithms, regularization methods (dropout, data augmentation, etc.), number of layers/parameters, and performance obtained with your model on the train/valid/test data sets, including the plots of the evolution of losses and accuracy.
* It would also be very valuable your feedback on the use of “Cesvima” or “Google Colab" services.

In the submission via Moodle, attach your Python (.py) or Jupyter Notebook (.ipynb) source file, including in the report all results of computations attached to the code that generated them.

The assignment must be done in groups of 3 students.

In [11]:
import math
from sklearn.metrics import confusion_matrix
from tensorflow.keras.saving import load_model

def show_test_results(model):
    y_true, y_pred = test_net(model, anns_test)
    # Compute the confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(categories.values()))
    draw_confusion_matrix(cm, categories)
    print_results(cm)

def training_loop(net_type):
    # Training loop
    for i in range(num_models):
        N_steps = (N_train/b_size[i]) * N_epochs[i]
        if isinstance(l_rate[i], list): # Si es cosine decay
            lr = get_lr_schedule(N_steps, l_rate[i][0], l_rate[i][1], l_rate[i][2], l_rate[i][3])
        else:
            lr = l_rate[i]

        if(net_type == 'ffNN'):
            model, h = train_ffnn(lr, b_size[i], N_epochs[i], optimizer[i], opt_hparams[i], 
                             neurons_hidden_layers[i], activ_fun[i], regularization[i], reg_param[i], 
                                  batch_norm[i], patience_lr[i], patience_es[i],
                                  class_weight[i],metric)

        elif(net_type == 'CNN'):
            model, h = train_cnn(cnn_structure[i], lr, b_size[i], N_epochs[i], optimizer[i], opt_hparams[i], 
                             neurons_hidden_layers[i], activ_fun[i], regularization[i], reg_param[i], 
                                 batch_norm[i], patience_lr[i], patience_es[i], 
                                 class_weight[i], metric)

        if type(metric[0]) != str:
            val_indexer = 'val_'+metric[0].name
        else:
            val_indexer = 'val_'+metric[0]

        best_idx = int(np.argmax(h.history[val_indexer]))
        best_value = np.max(h.history[val_indexer])
        print('Best validation model: epoch ' + str(best_idx+1), ' - '+val_indexer+': ' + str(best_value))
        show_test_results(model)
        

In [ ]:
net_type = 'CNN
'
if(net_type == 'CNN' or net_type == 'ffNN'):
    training_loop(net_type)
    
elif(net_type == 'LOAD'):
        model = load_model('/kaggle/working/2023-11-02_14-14-41.keras')
        show_test_results(model)

Warmup 0 Decay 15825
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 224, 224, 64)      1792      
                                                                 
 activation (Activation)     (None, 224, 224, 64)      0         
                                                                 
 conv2d_1 (Conv2D)           (None, 224, 224, 64)      36928     
                                                                 
 activation_1 (Activation)   (None, 224, 224, 64)      0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 112, 112, 64)     0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 112, 112, 128)     73856     
                                   